<a href="https://colab.research.google.com/github/kachionyenokwe/TempSound/blob/main/notebooks/01_yamnet_sed_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install yt-dlp for audio fetching and TensorFlow Hub for YAMNet
!pip install -q yt-dlp tensorflow-hub librosa soundfile

import os
import json
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import librosa
import soundfile as sf
import yt-dlp

print("✓ Environment dependencies loaded successfully.")
print("TensorFlow Version:", tf.__version__)

In [ ]:
# Load pre-trained YAMNet model from TensorFlow Hub
print("[*] Loading YAMNet model from TFHub...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

# Load YAMNet class mapping (521 AudioSet event classes)
class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
class_names = []
with tf.io.gfile.Gfile(class_map_path) as csv_file:
    import csv
    reader = csv.DictReader(csv_file)
    for row in reader:
        class_names.append(row['display_name'])

class_names = np.array(class_names)
print(f"[✓] YAMNet loaded successfully with {len(class_names)} audio event classes.")

In [ ]:
# Create working directories
os.makedirs("audio_downloads", exist_ok=True)
os.makedirs("output", exist_ok=True)

# Option A: Upload manually if running standalone in Colab
from google.colab import files
if not os.path.exists("gold_standard_50_sed_timestamps.json"):
    print("[*] Please upload 'gold_standard_50_sed_timestamps.json':")
    uploaded = files.upload()

with open("gold_standard_50_sed_timestamps.json", "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"[✓] Loaded manifest with {len(manifest)} Gold Standard prompts.")

In [ ]:
def download_audio_clip(youtube_id, output_dir="audio_downloads"):
    """
    Downloads audio waveform for a given YouTube ID and converts to 16kHz mono WAV.
    """
    output_path = os.path.join(output_dir, f"{youtube_id}.wav")
    if os.path.exists(output_path):
        return output_path

    url = f"https://www.youtube.com/watch?v={youtube_id}"
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'outtmpl': os.path.join(output_dir, f"{youtube_id}.%(ext)s"),
        'quiet': True,
        'no_warnings': True
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

        # Resample to 16kHz mono as required by YAMNet
        if os.path.exists(output_path):
            audio, sr = librosa.load(output_path, sr=16000, mono=True)
            sf.write(output_path, audio, 16000)
            return output_path
    except Exception as e:
        print(f"[Warning] Failed to download {youtube_id}: {e}")
        return None

In [ ]:
CONFIDENCE_THRESHOLD = 0.20  # Minimum confidence score for detected event

def process_audio_with_yamnet(wav_path):
    """
    Executes YAMNet frame inference and extracts event onset/offset timestamps.
    YAMNet uses 0.96s frames sliding every 0.48s.
    """
    audio_data, sr = librosa.load(wav_path, sr=16000, mono=True)

    # YAMNet inference
    scores, embeddings, spectrogram = yamnet_model(audio_data)
    scores = scores.numpy()  # Shape: (N_frames, 521)

    detected_events = []
    num_frames = scores.shape[0]

    for frame_idx in range(num_frames):
        # YAMNet frame timing: 0.48s step size
        t_start = round(frame_idx * 0.48, 2)
        t_end = round(t_start + 0.96, 2)

        top_class_idx = np.argmax(scores[frame_idx])
        confidence = float(scores[frame_idx][top_class_idx])

        if confidence >= CONFIDENCE_THRESHOLD:
            label = class_names[top_class_idx]

            # Avoid duplicate consecutive frame logging for the same class
            if detected_events and detected_events[-1]["label"] == label:
                detected_events[-1]["t_end"] = t_end
            else:
                detected_events.append({
                    "label": label,
                    "confidence": round(confidence, 3),
                    "t_start": t_start,
                    "t_end": t_end
                })

    return detected_events

In [ ]:
print("[*] Starting batch SED timestamp extraction...")

for entry in manifest:
    yt_id = entry.get("youtube_id")
    sample_id = entry.get("sample_id")
    caption = entry.get("caption")

    print(f"\nProcessing Sample {sample_id}/50: '{caption}'")

    # 1. Fetch Audio
    wav_file = download_audio_clip(yt_id)

    if wav_file and os.path.exists(wav_file):
        # 2. Extract SED Timestamps
        events = process_audio_with_yamnet(wav_file)
        entry["detected_events"] = events
        print(f" -> Extracted {len(events)} event intervals.")
    else:
        print(f" -> [Skipped] Audio unavailable for YouTube ID: {yt_id}")

# Export populated ground-truth manifest
output_manifest_path = "output/gold_standard_50_sed_timestamps_populated.json"
with open(output_manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print(f"\n[✓] SED Extraction Complete!")
print(f"[✓] Populated manifest saved to: {output_manifest_path}")

# Trigger automatic download back to host machine
files.download(output_manifest_path)